# 11 — Risk Module Quickstart

R-multiple position sizing: stop placement, position planning, and regime-aware risk scaling. See the [risk README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/risk/README.md) for full documentation.

In [ ]:
from __future__ import annotations
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

from swing_screener.risk import RiskConfig, compute_stop, position_plan, build_trade_plans, compute_regime_risk_multiplier
from swing_screener.data.providers import get_market_data_provider
from swing_screener.indicators.volatility import VolatilityConfig, compute_volatility_features

## Basic Stop & Position Plan

Single-stock risk calculation: stop = entry − k_atr × ATR14. Shares sized to risk budget (account_size × risk_pct) and capped at max_position_pct.

In [ ]:
cfg = RiskConfig(
    account_size=50000,
    account_currency="USD",
    risk_pct=0.01,
    k_atr=2.0,
    min_rr=2.0,
)
stop = compute_stop(entry=175.50, atr14=3.25, k_atr=2.0)
plan = position_plan(entry=175.50, atr14=3.25, cfg=cfg)
print(f"Entry: 175.50, Stop: {stop:.2f}, 1R: {175.50 - stop:.2f}")
print(f"Shares: {plan['shares']}, Risk: ${plan['realized_risk']:.2f}")

## Build Trade Plans for a Universe

Fetch OHLCV for a few tickers, compute ATR14 features, build synthetic signal board, then generate trade plans via `build_trade_plans`.

In [ ]:
provider = get_market_data_provider()
ohlcv = provider.fetch_ohlcv(["SPY", "AAPL", "MSFT"], "2023-01-01", "2024-12-31")
print(f"OHLCV shape: {ohlcv.shape}")
ohlcv.head()

In [ ]:
# ATR14 per ticker
vol_feats = compute_volatility_features(ohlcv, VolatilityConfig(atr_window=14))
print("Volatility features:")
vol_feats

In [ ]:
# Build ranked_universe: must have atr14, last (entry), and currency columns
last_close = ohlcv["Close"].iloc[-1]
ranked = vol_feats.copy()
ranked["last"] = last_close
ranked["currency"] = "USD"
ranked

In [ ]:
# Build a synthetic signal board
signal_board = pd.DataFrame({
    "last": last_close,
    "signal": "breakout",
}, index=pd.Index(last_close.index, name="ticker"))
signal_board

In [ ]:
# Generate trade plans — only tickers with signal != 'none' are processed
plans = build_trade_plans(ranked, signal_board, cfg)
plans

Each plan row shows entry, stop, shares, position value, and realized risk. The sizing respects both risk budget (1% of account) and max position cap (60% of account).

## Regime-Aware Risk Scaling

Compute a market-wide risk multiplier by checking benchmark trend (price vs SMA) and volatility (ATR%). When regime scaling is enabled, the multiplier is applied before position sizing.

In [ ]:
# Fetch longer SPY history for SMA200 + ATR checks
spy_ohlcv = provider.fetch_ohlcv(["SPY"], "2022-01-01", "2024-12-31")

regime_cfg = RiskConfig(
    account_size=50000,
    regime_enabled=True,
    regime_trend_sma=200,
    regime_trend_multiplier=0.5,
    regime_vol_atr_window=14,
    regime_vol_atr_pct_threshold=0.5,
    regime_vol_multiplier=0.5,
)

multiplier, details = compute_regime_risk_multiplier(spy_ohlcv, "SPY", regime_cfg)
print(f"Risk multiplier: {multiplier}")
print(f"Trend below SMA200: {details['trend_below_sma']}")
print(f"ATR%: {details['atr_pct']}")
print(f"Reasons: {details['reasons']}")